# Ray: **4 logical GPU nodes** + **Master** + Redis **PEL/XAUTOCLAIM** fault tolerance

**Separated architecture**
- **AWS VPS (control plane)**: **nginx** load balancer (**`least_conn`**) → **FastAPI master**; **Redis** message broker (**Streams**) on the same VPC (bind private/loopback).
- **GPU cluster (Google Colab)**: **Inference + RAG + Ollama** only — **no** nginx/redis on Colab.
- **Cloudflare Tunnel (`cloudflared` on VPS)**: Colab talks to **`BASE_URL`** / **`REDIS_URL`** via **tunnel hostnames** through Cloudflare (TLS), not by punching raw **`public-ip:6379`**. Configure TCP for Redis per Cloudflare docs; see **`deploy/CLOUDFLARE_TUNNEL.md`** and **`deploy/cloudflared/config.example.yml`**.

This notebook expects a reachable **`REDIS_URL`** to that VPS broker — paste the tunnel (or VPN) URL in section **3** when **`USE_LOCAL_REDIS=False`**.

**What this simulates**
- **Broker**: Redis Streams + consumer group `workers` (same as your production codebase).
- **Master (Ray actor)**: `XADD` via `enqueue`; `wait_result`; **PEL snapshots** (`XPENDING`).
- **4× Inference nodes (Ray actors)**: **each owns** a **daemon thread** running `XREADGROUP` + **`XAUTOCLAIM`** (idle pending → reclaim / **task reassignment**) + **private FAISS RAG dir** (`rag_sim/node_<i>`).
- **LLM**: each node calls **`OLLAMA_BASE_URL`** (typically one shared `ollama serve`; processes are still logically separate workers).

**Artifacts**
- All **models / Redis stream ops / RAG / HTTP LLM helpers** live in **`cluster_sim_standalone.py`** alongside this notebook (imported below). Ray actors are defined **in-notebook** so you can tweak scheduling.

**Sections**: setup → **broker `REDIS_URL` (VPS or local)** → Ollama → env → Ray cluster → demos → fault injection → cleanup.

---
**Tip**: shorten `REDIS_CLAIM_IDLE_MS` (e.g. `8000`) in the env cell before starting nodes so stalled PEL entries reclaim quickly after `ray.kill()`.

## 1) Project root & GPU probe

In [ ]:
import os, sys, subprocess

!nvidia-smi -L || echo "No nvidia-smi (CPU Ray ok)"

# !git clone https://github.com/YOU/REPO.git /content/dist
PROJECT_ROOT = "/content/dist"
assert os.path.isfile(os.path.join(PROJECT_ROOT, "main.py")), "clone/unzip repo to PROJECT_ROOT"

_colab_dir = os.path.join(PROJECT_ROOT, "notebooks", "colab")
assert os.path.isfile(os.path.join(_colab_dir, "cluster_sim_standalone.py")), "missing cluster_sim_standalone.py"

for p in (PROJECT_ROOT, _colab_dir):
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTHONPATH"] = _colab_dir + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PROJECT_ROOT=", PROJECT_ROOT)

## 2) Packages: Redis, **Ray**, embeddings, HTTP

In [ ]:
import subprocess

subprocess.run(
    ["bash", "-lc", "sudo apt-get update -qq && sudo apt-get install -y -qq pciutils zstd curl"],
    check=True,
)
# Redis server packages are installed only when USE_LOCAL_REDIS (next section).

In [ ]:
%pip install -q "ray[default]>=2.9" redis faiss-cpu sentence-transformers requests numpy python-dotenv

## 3) **Message broker** — Redis on **AWS VPS** (or optional local Redis)

- **Default / production**: set **`USE_LOCAL_REDIS = False`**, paste your VPS **`REDIS_URL`** (from AWS EC2/Elastiache / private DNS). Script runs **`PING`** to verify connectivity from Colab.
- **Offline demo**: **`USE_LOCAL_REDIS = True`** installs and starts Redis on this runtime (same host as Ray — not representative of VPS broker, but works without AWS).

In [ ]:
import subprocess

subprocess.run(["bash", "-lc", "sudo redis-server --daemonize yes && sleep 1 && redis-cli ping"], check=True)

## 4) **Ollama** (shared inference server)

In [ ]:
import subprocess, time

subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
_o = open("/content/ollama.log","w")
_e = open("/content/ollama_err.log","w")
subprocess.Popen(["ollama","serve"], stdout=_o, stderr=_e, start_new_session=True)
time.sleep(12)
subprocess.run(["ollama","--version"], check=True)

In [ ]:
import os, subprocess, sys
_m = os.environ.get("OLLAMA_MODEL","llama3.2:1b")
p = subprocess.Popen(["ollama","pull",_m], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
assert p.stdout
for line in p.stdout:
    print(line, end=""); sys.stdout.flush()
assert p.wait() == 0

## 5) Environment + import **AI / broker** module

`cluster_sim_standalone.py` contains **TaskRequest/Response**, **Redis XADD/XREADGROUP/XAUTOCLAIM**, **FaissRetriever**, **Ollama chat**.

In [ ]:
import importlib.util, os, pathlib, sys

assert os.environ.get("REDIS_URL"), "Run section 3 first — REDIS_URL must point to AWS VPS Redis (or local)."
os.environ.setdefault("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_MODEL", "llama3.2:1b")
os.environ.setdefault("REDIS_CLAIM_IDLE_MS", "12000")
os.environ.setdefault("REDIS_BLOCK_MS", "3000")
os.environ.setdefault("RAG_TOP_K", "4")

_SIM_PATH = pathlib.Path(PROJECT_ROOT) / "notebooks/colab/cluster_sim_standalone.py"
_spec = importlib.util.spec_from_file_location("cluster_sim_standalone", str(_SIM_PATH))
assert _spec and _spec.loader
sim = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(sim)

RAG_CLUSTER_ROOT = pathlib.Path("/tmp/rag_sim_cluster")
RAG_CLUSTER_ROOT.mkdir(parents=True, exist_ok=True)

print("Loaded sim module from", _SIM_PATH)
print("PEL reclaim idle ms", os.environ["REDIS_CLAIM_IDLE_MS"])

## 6) **Ray** init + **InferenceNode** (4× logical GPU worker) + **MasterCoordinator**

- **GPU**: assigns `num_gpus=0.25` × 4 on a **1‑GPU** Colab VM (scheduling abstraction). Fallback: **CPU actors** (`num_gpus=0`).
- **Fault model**: destroying an actor frees its runtime; unfinished stream entries stay **PEL pending** → **XAUTOCLAIM** after idle threshold.

In [ ]:
import os, pathlib, threading, time

import ray

_redis_url = os.environ["REDIS_URL"]
_obase = os.environ["OLLAMA_BASE_URL"]
_omodel = os.environ["OLLAMA_MODEL"]


try:
    ray.shutdown()
except Exception:
    pass

try:
    ray.init(num_cpus=10, num_gpus=1, ignore_reinit_error=True)
    _gpu = float(ray.cluster_resources().get("GPU", 0) or ray.cluster_resources().get("gpu", 0) or 0)
    if _gpu >= 1.0:
        _resources = {"num_cpus": 2, "num_gpus": 0.25}
    else:
        raise RuntimeError("no GPU resource")
except Exception:
    try:
        ray.shutdown()
    except Exception:
        pass
    ray.init(num_cpus=10, num_gpus=0, ignore_reinit_error=True)
    _resources = {"num_cpus": 2, "num_gpus": 0}


class InferenceNodeImpl:
    """Logical GPU/RAG/Ollama consumer; thread runs Streams loop."""

    def __init__(self, node_index: int):
        import cluster_sim_standalone as _sim

        self.node_index = int(node_index)
        self.consumer_name = f"gpu-node-{self.node_index}"
        self.rag_dir = str(pathlib.Path(RAG_CLUSTER_ROOT) / f"node_{self.node_index}")
        pathlib.Path(self.rag_dir).mkdir(parents=True, exist_ok=True)

        self._stop = threading.Event()
        self._th = threading.Thread(
            target=_sim.thread_consumer_forever,
            args=(
                _redis_url,
                self.consumer_name,
                self.consumer_name,
                self.rag_dir,
                self._stop,
                _obase,
                _omodel,
            ),
            daemon=True,
            name=f"stream-{self.consumer_name}",
        )
        self._th.start()

    def ping(self) -> str:
        return self.consumer_name

    def shutdown(self):
        self._stop.set()


class MasterCoordinatorImpl:
    """Control plane orchestration surfaced as a Ray actor."""

    def __init__(self, redis_url: str):
        self._redis_url = redis_url

    def submit_task(self, query: str, metadata: dict | None = None):
        import cluster_sim_standalone as _sim

        r = _sim.redis_connect(self._redis_url)
        _sim.ensure_consumer_group(r)
        return _sim.enqueue_task(r, _sim.TaskRequest(query=query, metadata=metadata or {}))

    def wait_result(self, task_id: str, timeout_sec: float = 240.0):
        import cluster_sim_standalone as _sim

        return _sim.poll_task_until_done(self._redis_url, task_id, timeout_sec=timeout_sec)

    def pel_snapshot(self):
        import cluster_sim_standalone as _sim

        r = _sim.redis_connect(self._redis_url)
        _sim.ensure_consumer_group(r)
        return _sim.xpending_summary(r)

    def stream_len(self) -> int:
        import cluster_sim_standalone as _sim

        r = _sim.redis_connect(self._redis_url)
        try:
            return int(r.xlen(_sim.STREAM_KEY))
        except Exception:
            return -1


InferenceNode = ray.remote(**_resources)(InferenceNodeImpl)
MasterCoordinator = ray.remote(num_cpus=1)(MasterCoordinatorImpl)

## 7) Spawn **Master** + **4 nodes** ; verify consumers

In [ ]:
master = MasterCoordinator.remote(_redis_url)
nodes = [InferenceNode.remote(i) for i in range(4)]
print("Consumers:", ray.get([n.ping.remote() for n in nodes]))
print("PEL (idle):", ray.get(master.pel_snapshot.remote()))

## 8) Load demo: enqueue via **Master** Ray actor

In [ ]:
import ray

_queries = [
    "Explain Redis Streams PEL briefly.",
    "How does Ray simulate distributed GPUs?",
    "What is retrieval-augmented generation?",
    "Describe least_conn load balancing.",
]
_ids = ray.get([master.submit_task.remote(q) for q in _queries])
print("task ids:", _ids)

_answers = []
for tid in _ids:
    row = ray.get(master.wait_result.remote(tid))
    _answers.append((tid, row.get("status"), (row.get("answer") or "")[:280]))

for row in _answers:
    print("\n===", row[0], "status=", row[1], "===\n", row[2])

## 9) **Fault injection** — kill node `gpu-node-1`, enqueue, watch **reclaim**

After **`ray.kill(nodes[1])`**, unfinished work for that consumer may sit in PEL → other nodes’ **`XAUTOCLAIM`** (**idle** passes `REDIS_CLAIM_IDLE_MS`) recover it.
**Respawn** a replacement actor occupying the same numeric slot if you still want four live consumers.

In [ ]:
import time, ray

print("PEL before kill:", ray.get(master.pel_snapshot.remote()))

ray.kill(nodes[1])
print("Killed gpu-node-1 actor")
time.sleep(2)

_fault_tid = ray.get(master.submit_task.remote("Write one sentence about fault tolerance after a worker crashes."))
print("new task", _fault_tid)

# Optional: shorten idle wait by temporarily lowering reclaim threshold in **new replacement** actors only affects their threads — idle ms is read at loop time via os.environ:
os.environ["REDIS_CLAIM_IDLE_MS"] = "6000"
_replacement = InferenceNode.remote(91)  # new consumer gpu-node-91 joins reclaim herd

_res = ray.get(master.wait_result.remote(_fault_tid, timeout_sec=240))
print("result status", _res.get("status"), "answer snippet", (_res.get("answer") or "")[:320])
print("PEL after:", ray.get(master.pel_snapshot.remote()))

nodes[1] = _replacement

## 10) Cleanup (best-effort)

Stop Ray; interrupt kernel to exit Ollama/Redis daemon processes or restart runtime.

In [ ]:
import ray

try:
    for n in nodes:
        try:
            ray.get(n.shutdown.remote())
        except Exception:
            pass
except Exception:
    pass
ray.kill(master)
try:
    [ray.kill(n) for n in nodes]
except Exception:
    pass
try:
    ray.shutdown()
except Exception:
    pass
print("Ray shutdown dispatched.")